# TF-IDF + Logistic Regression + sublinear_tf=True

Experimento H - Divisão do treino em duas etapas

In [26]:
import pandas as pd
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

## 1. Carregamento dos dados

In [ ]:
df = pd.read_csv("../data/train_limpo.csv") # Conjunto de treino limpo

Y = df["clarity"]

print(df.shape)
df.head()

(20092, 2)


,resp_text,clarity
0,prezado a senhor a esclarecemos que o serviço ...,c5
1,prezada cidadã as informações sobre óbitos no ...,c1
2,prezado senhor julio a ouvidoria geral da petr...,c1
3,prezado a senhor a esclarecemos que o serviço ...,c234
4,senhor o serviço de informações ao cidadão do ...,c234


## 2. Separação dos dados

In [29]:
x_train_text, x_test_text, y_train, y_test = train_test_split(
    df["resp_text"],
    Y,
    test_size=0.2,
    random_state=123,
    stratify=Y
)

print("Treino:", x_train_text.shape)
print("Teste:", x_test_text.shape)

Treino: (16073,)
Teste: (4019,)


## 3. Vetorização TF-IDF

Neste experimento, utilizamos `sublinear_tf=True`, que aplica uma ponderação sublinear à frequência dos termos.

In [30]:
vect = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

x_train = vect.fit_transform(x_train_text)
x_test = vect.transform(x_test_text)

print("Treino TF-IDF:", x_train.shape)
print("Teste TF-IDF:", x_test.shape)

Treino TF-IDF: (16073, 1196293)
Teste TF-IDF: (4019, 1196293)


## 4.1. Treino da Regressão Logística em duas etapas
Experimento H1:

ETAPA 1: C234 versus (C1 + C5)

ETAPA 2: C1 versus C5

In [31]:
# # ============================================================
# # EXPERIMENTO H1 — CLASSIFICAÇÃO EM DUAS ETAPAS
# # ETAPA 1: C234 versus (C1 + C5)
# # ETAPA 2: C1 versus C5
# # ============================================================

# # Padronizar os rótulos
# y_train = y_train.astype(str).str.lower().str.strip()
# y_test = y_test.astype(str).str.lower().str.strip()

# # ------------------------------------------------------------
# # ETAPA 1: C234 versus (C1 + C5)
# # ------------------------------------------------------------

# # Criar rótulos binários para o primeiro classificador
# y_train_etapa1 = y_train.apply(
#     lambda classe: "c234" if classe == "c234" else "outros"
# )

# clf1 = LogisticRegression(
#     C=1,
#     class_weight="balanced",
#     max_iter=1000
# )

# clf1.fit(x_train, y_train_etapa1)

# # ------------------------------------------------------------
# # ETAPA 2: C1 versus C5
# # ------------------------------------------------------------

# # Selecionar somente exemplos C1 e C5 do treino
# mask_etapa2 = y_train.isin(["c1", "c5"])

# # Convertendo a máscara para array do NumPy (.values) para evitar o erro no SciPy
# x_train_etapa2 = x_train[mask_etapa2.values]
# y_train_etapa2 = y_train[mask_etapa2]

# clf2 = LogisticRegression(
#     C=1,
#     class_weight="balanced",
#     max_iter=1000
# )

# clf2.fit(x_train_etapa2, y_train_etapa2)

# print("Classificador 1 treinado: C234 x (C1 + C5)")
# print("Classificador 2 treinado: C1 x C5")
# print("\nQuantidade de exemplos na etapa 2:")
# print(y_train_etapa2.value_counts())

## 4.2 Treino da Regressão Logística em duas etapas (melhor)
Experimento H2:

ETAPA 1: C5 versus (C1 + C234)

ETAPA 2: C1 versus C234

In [ ]:
# ============================================================
# EXPERIMENTO H2 — CLASSIFICAÇÃO EM DUAS ETAPAS
# ETAPA 1: C5 versus (C1 + C234)
# ETAPA 2: C1 versus C234
# ============================================================

# Padronizar os rótulos
y_train = y_train.astype(str).str.lower().str.strip()
y_test = y_test.astype(str).str.lower().str.strip()

# ------------------------------------------------------------
# ETAPA 1: C5 versus (C1 + C234)
# ------------------------------------------------------------

y_train_etapa1 = y_train.apply(
    lambda classe: "c5" if classe == "c5" else "outros"
)

clf1 = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf1.fit(x_train, y_train_etapa1)

# ------------------------------------------------------------
# ETAPA 2: C1 versus C234
# ------------------------------------------------------------

mask_etapa2 = y_train.isin(["c1", "c234"])

x_train_etapa2 = x_train[mask_etapa2.values]
y_train_etapa2 = y_train[mask_etapa2]

clf2 = LogisticRegression(
    C=1,
    class_weight="balanced",
    max_iter=1000
)

clf2.fit(x_train_etapa2, y_train_etapa2)

print("Classificador 1 treinado: c5 x (c1 + c234)")
print("Classificador 2 treinado: c1 x c234")

print("\nQuantidade de exemplos na etapa 2:")
print(y_train_etapa2.value_counts())

Classificador 1 treinado: C5 x (C1 + C234)
Classificador 2 treinado: C1 x C234

Quantidade de exemplos na etapa 2:
clarity
c234    5482
c1      5078
Name: count, dtype: int64


## 5.1 Avaliação Experimento H1

In [33]:
# # ============================================================
# # H1 - PREDIÇÃO EM DUAS ETAPAS
# # ============================================================

# # Primeira etapa
# pred_etapa1 = clf1.predict(x_test)

# # Inicializar as previsões finais
# predicted = pd.Series(
#     index=y_test.index,
#     dtype="object"
# )

# # Quem foi classificado como C234 já recebe a classe final
# mask_c234 = pred_etapa1 == "c234"
# predicted.loc[mask_c234] = "c234"

# # Quem foi classificado como "outros" segue para a etapa 2
# mask_outros = pred_etapa1 == "outros"

# # Segunda etapa
# if mask_outros.sum() > 0:
#     predicted.loc[mask_outros] = clf2.predict(
#         x_test[mask_outros]
#     )

# # Garantir que os rótulos estejam alinhados
# predicted = predicted.loc[y_test.index]

# # Métricas finais
# score = f1_score(
#     y_test,
#     predicted,
#     average="macro"
# )

# accuracy = accuracy_score(
#     y_test,
#     predicted
# )

# print("RESULTADO FINAL — DUAS ETAPAS")
# print(f"Macro F1: {score:.4f}")
# print(f"Accuracy: {accuracy:.4f}")

## 5.2 Avaliação Experimento H2

In [34]:
# ============================================================
# H2 - PREDIÇÃO EM DUAS ETAPAS
# ============================================================

# Primeira etapa
pred_etapa1 = clf1.predict(x_test)

# Inicializar previsões finais
predicted = pd.Series(
    index=y_test.index,
    dtype="object"
)

# Quem foi classificado como C5 recebe a classe final
mask_c5 = pred_etapa1 == "c5"
predicted.loc[mask_c5] = "c5"

# Quem foi classificado como "outros" segue para a etapa 2
mask_outros = pred_etapa1 == "outros"

# Segunda etapa
if mask_outros.sum() > 0:
    predicted.loc[mask_outros] = clf2.predict(
        x_test[mask_outros]
    )

# Garantir alinhamento
predicted = predicted.loc[y_test.index]

# Métricas finais
score = f1_score(
    y_test,
    predicted,
    average="macro"
)

accuracy = accuracy_score(
    y_test,
    predicted
)

print("RESULTADO FINAL — DUAS ETAPAS H2")
print(f"Macro F1: {score:.4f}")
print(f"Accuracy: {accuracy:.4f}")

RESULTADO FINAL — DUAS ETAPAS H2
Macro F1: 0.4782
Accuracy: 0.4807


In [35]:
from sklearn.metrics import classification_report, confusion_matrix

print("RELATÓRIO DE CLASSIFICAÇÃO")
print(
    classification_report(
        y_test,
        predicted,
        labels=["c1", "c234", "c5"],
        zero_division=0
    )
)

print("MATRIZ DE CONFUSÃO")
print(
    confusion_matrix(
        y_test,
        predicted,
        labels=["c1", "c234", "c5"]
    )
)

RELATÓRIO DE CLASSIFICAÇÃO
              precision    recall  f1-score   support

          c1       0.51      0.51      0.51      1269
        c234       0.41      0.36      0.39      1371
          c5       0.51      0.57      0.54      1379

    accuracy                           0.48      4019
   macro avg       0.48      0.48      0.48      4019
weighted avg       0.48      0.48      0.48      4019

MATRIZ DE CONFUSÃO
[[648 324 297]
 [401 495 475]
 [212 378 789]]


## 6. Validação Cruzada - Experimento H2

In [39]:

# ============================================================
# EXPERIMENTO H2 — VALIDAÇÃO CRUZADA EM DUAS ETAPAS
# C5 x (C1 + C234) -> C1 x C234
# ============================================================

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Dados
# ------------------------------------------------------------

df = pd.read_csv("train_limpo.csv")

X = df["resp_text"].reset_index(drop=True)

y = (
    df["clarity"]
    .astype(str)
    .str.lower()
    .str.strip()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 2. Validação cruzada estratificada
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

resultados_f1 = []
resultados_acc = []

resultados_f1_treino = []
resultados_acc_treino = []

y_real_completo = []
y_pred_completo = []

# ------------------------------------------------------------
# 3. Executar os folds
# ------------------------------------------------------------

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):

    print(f"\n{'='*50}")
    print(f"FOLD {fold}")
    print(f"{'='*50}")

    X_train_text = X.iloc[train_idx]
    X_val_text = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # --------------------------------------------------------
    # TF-IDF
    # --------------------------------------------------------

    vect = TfidfVectorizer(
        analyzer="word",
        sublinear_tf=True,
        ngram_range=(1, 3)
    )

    x_train = vect.fit_transform(X_train_text)
    x_val = vect.transform(X_val_text)

    # --------------------------------------------------------
    # ETAPA 1: C5 versus (C1 + C234)
    # --------------------------------------------------------

    y_train_etapa1 = y_train.apply(
        lambda classe: "c5" if classe == "c5" else "outros"
    )

    clf1 = LogisticRegression(
        C=1,
        class_weight="balanced",
        max_iter=1000
    )

    clf1.fit(x_train, y_train_etapa1)

    # --------------------------------------------------------
    # ETAPA 2: C1 versus C234
    # --------------------------------------------------------

    mask_etapa2 = y_train.isin(["c1", "c234"])

    x_train_etapa2 = x_train[mask_etapa2.to_numpy()]
    y_train_etapa2 = y_train[mask_etapa2]

    clf2 = LogisticRegression(
        C=1,
        class_weight="balanced",
        max_iter=1000
    )

    clf2.fit(x_train_etapa2, y_train_etapa2)

    # --------------------------------------------------------
    # PREDIÇÃO ENCADEADA NO TREINO
    # --------------------------------------------------------

    pred_treino_etapa1 = clf1.predict(x_train)

    pred_treino = np.empty(len(train_idx), dtype=object)

    mask_c5_treino = pred_treino_etapa1 == "c5"
    pred_treino[mask_c5_treino] = "c5"

    mask_outros_treino = pred_treino_etapa1 == "outros"

    if mask_outros_treino.sum() > 0:
        pred_treino[mask_outros_treino] = clf2.predict(
            x_train[mask_outros_treino]
        )

    # --------------------------------------------------------
    # PREDIÇÃO ENCADEADA NA VALIDAÇÃO
    # --------------------------------------------------------

    pred_etapa1 = clf1.predict(x_val)

    predicted = np.empty(len(val_idx), dtype=object)

    mask_c5 = pred_etapa1 == "c5"
    predicted[mask_c5] = "c5"

    mask_outros = pred_etapa1 == "outros"

    if mask_outros.sum() > 0:
        predicted[mask_outros] = clf2.predict(
            x_val[mask_outros]
        )

    # --------------------------------------------------------
    # MÉTRICAS DO TREINO
    # --------------------------------------------------------

    f1_treino = f1_score(
        y_train,
        pred_treino,
        average="macro"
    )

    acc_treino = accuracy_score(
        y_train,
        pred_treino
    )

    resultados_f1_treino.append(f1_treino)
    resultados_acc_treino.append(acc_treino)

    # --------------------------------------------------------
    # MÉTRICAS DA VALIDAÇÃO
    # --------------------------------------------------------

    f1 = f1_score(
        y_val,
        predicted,
        average="macro"
    )

    acc = accuracy_score(
        y_val,
        predicted
    )

    resultados_f1.append(f1)
    resultados_acc.append(acc)

    y_real_completo.extend(y_val.tolist())
    y_pred_completo.extend(predicted.tolist())

    print(f"Macro-F1 Treino: {f1_treino:.4f}")
    print(f"Acurácia Treino: {acc_treino:.4f}")
    print(f"Macro-F1 Validação: {f1:.4f}")
    print(f"Acurácia Validação: {acc:.4f}")


# ------------------------------------------------------------
# 4. MÉDIA E DESVIO PADRÃO
# ------------------------------------------------------------

print("\n" + "="*50)
print("RESULTADO DA VALIDAÇÃO CRUZADA — H2")
print("="*50)

print("\nDESEMPENHO MÉDIO NO TREINO")

print(
    f"Macro-F1: {np.mean(resultados_f1_treino):.4f} "
    f"± {np.std(resultados_f1_treino):.4f}"
)

print(
    f"Acurácia: {np.mean(resultados_acc_treino):.4f} "
    f"± {np.std(resultados_acc_treino):.4f}"
)

print("\nDESEMPENHO MÉDIO NA VALIDAÇÃO")

print(
    f"Macro-F1: {np.mean(resultados_f1):.4f} "
    f"± {np.std(resultados_f1):.4f}"
)

print(
    f"Acurácia: {np.mean(resultados_acc):.4f} "
    f"± {np.std(resultados_acc):.4f}"
)

# ------------------------------------------------------------
# 5. RELATÓRIO OUT-OF-FOLD
# ------------------------------------------------------------

print("\nRELATÓRIO DE CLASSIFICAÇÃO — OUT-OF-FOLD")

print(
    classification_report(
        y_real_completo,
        y_pred_completo,
        labels=["c1", "c234", "c5"],
        zero_division=0
    )
)

print("MATRIZ DE CONFUSÃO — OUT-OF-FOLD")

print(
    confusion_matrix(
        y_real_completo,
        y_pred_completo,
        labels=["c1", "c234", "c5"]
    )
)


FOLD 1
Macro-F1 Treino: 0.7803
Acurácia Treino: 0.7815
Macro-F1 Validação: 0.4669
Acurácia Validação: 0.4713

FOLD 2
Macro-F1 Treino: 0.7887
Acurácia Treino: 0.7901
Macro-F1 Validação: 0.4653
Acurácia Validação: 0.4700

FOLD 3
Macro-F1 Treino: 0.7892
Acurácia Treino: 0.7905
Macro-F1 Validação: 0.4652
Acurácia Validação: 0.4676

FOLD 4
Macro-F1 Treino: 0.7801
Acurácia Treino: 0.7815
Macro-F1 Validação: 0.4463
Acurácia Validação: 0.4495

FOLD 5
Macro-F1 Treino: 0.7895
Acurácia Treino: 0.7908
Macro-F1 Validação: 0.4650
Acurácia Validação: 0.4676

RESULTADO DA VALIDAÇÃO CRUZADA — H2

DESEMPENHO MÉDIO NO TREINO
Macro-F1: 0.7856 ± 0.0044
Acurácia: 0.7869 ± 0.0044

DESEMPENHO MÉDIO NA VALIDAÇÃO
Macro-F1: 0.4618 ± 0.0078
Acurácia: 0.4652 ± 0.0080

RELATÓRIO DE CLASSIFICAÇÃO — OUT-OF-FOLD
              precision    recall  f1-score   support

          c1       0.49      0.48      0.48      6347
        c234       0.40      0.35      0.37      6853
          c5       0.49      0.57      0.53  

## 6. Treinamento do modelo final com modelo H2

In [5]:
# ============================================================
# EXPERIMENTO H2 — TREINAMENTO FINAL COM TODOS OS DADOS
# ============================================================

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Carregar todos os dados de treino
df = pd.read_csv("../data/train_limpo.csv")

X_full = df["resp_text"]
y_full = df["clarity"]

# 2. Ajustar o TF-IDF com todos os textos
vect_H2 = TfidfVectorizer(
    analyzer="word",
    sublinear_tf=True,
    ngram_range=(1, 3)
)

X_full_tfidf = vect_H2.fit_transform(X_full)

# 3. Etapa 1: C5 vs demais classes
y_etapa1 = (y_full == "c5").astype(int)

clf_etapa1_H2 = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=1000
)

clf_etapa1_H2.fit(X_full_tfidf, y_etapa1)

# 4. Etapa 2: C1 vs C234
# Utilizar somente exemplos que não são C5
mask_etapa2 = y_full != "c5"

# Ajustado para usar .to_numpy() e evitar o erro com a matriz esparsa
X_etapa2 = X_full_tfidf[mask_etapa2.to_numpy()]
y_etapa2 = y_full[mask_etapa2]

clf_etapa2_H2 = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    max_iter=1000
)

clf_etapa2_H2.fit(X_etapa2, y_etapa2)

print("Experimento H2 treinado com todos os dados!")
print("Etapa 1: c5 vs demais classes")
print("Etapa 2: c1 vs c234")

Experimento H2 treinado com todos os dados!
Etapa 1: c5 vs demais classes
Etapa 2: c1 vs c234


## 7. Rotulação o conjunto de teste 1

In [6]:
# ============================================================
# EXPERIMENTO H2 — ROTULAÇÃO DO TESTE 1
# ============================================================

# 1. Carregar Teste 1
test_df = pd.read_csv("../data/test1_limpo.csv")

# 2. Transformar os textos usando o TF-IDF já treinado
X_test1_tfidf = vect_H2.transform(test_df["resp_text"])

# 3. Previsões da primeira etapa
pred_etapa1 = clf_etapa1_H2.predict(X_test1_tfidf)

# 4. Inicializar vetor de previsões finais
pred_H2 = []

for i, pred in enumerate(pred_etapa1):

    if pred == 1:
        # Primeira etapa identificou C5
        pred_H2.append("c5")

    else:
        # Segunda etapa decide entre C1 e C234
        pred_etapa2 = clf_etapa2_H2.predict(
            X_test1_tfidf[i]
        )[0]

        pred_H2.append(pred_etapa2)

# 5. Adicionar as previsões sem sobrescrever eventual rótulo real
test_df["clarity_pred_H2"] = pred_H2

# 6. Conferir distribuição das classes previstas
print("Distribuição das classes previstas:")
print(test_df["clarity_pred_H2"].value_counts())

print("\nPrimeiras previsões:")
display(test_df[["resp_text", "clarity_pred_H2"]].head())

Distribuição das classes previstas:
clarity_pred_H2
c5      344
c234    290
c1      266
Name: count, dtype: int64

Primeiras previsões:


,resp_text,clarity_pred_H2
0,prezada hellen dentre as pesquisas domiciliare...,c234
1,senhor francisco o serviço de informações ao c...,c234
2,prezada srª raylane em resposta a sua solicita...,c234
3,prezada senhora de acordo com o art 13 do decr...,c1
4,prezado a senhor a em atendimento ao pedido de...,c1


## 8. Salva o arquivo rotulado

In [7]:
# Salvar resultado
test_df.to_excel("../resultados/test1_rotulado_ep2_H2.xlsx", index=False)
print("Arquivo Teste 1 rotulado pelo H2 salvo!")

Arquivo Teste 1 rotulado pelo H2 salvo!


## 9. Avaliação do modelo H2

In [9]:
from sklearn.metrics import accuracy_score, classification_report

# Carregar os arquivos
ideal = pd.read_excel("../data/test1-rotulado-para-teste-estatistico.xlsx")
previsoes = pd.read_excel("../resultados/test1_rotulado_ep2_H2.xlsx")

# Conferir se os textos estão na mesma ordem
# assert ideal["resp_text"].equals(previsoes["resp_text"]), \
#     "Os textos não estão na mesma ordem!"

# Rótulos reais e previsões do modelo
y_real = ideal["clarity"]
y_pred = previsoes["clarity_pred_H2"]

classes = ["c1", "c234", "c5"]

# Calcular acurácia
acuracia = accuracy_score(y_real, y_pred)

print(f"Acurácia do novo modelo: {acuracia:.2%}")

# Relatório de classificação
print("\nRelatório de classificação:")
print(classification_report(
    y_real,
    y_pred,
    labels=classes,
    zero_division=0
))

Acurácia do novo modelo: 49.33%

Relatório de classificação:
              precision    recall  f1-score   support

          c1       0.53      0.47      0.49       300
        c234       0.44      0.42      0.43       300
          c5       0.51      0.59      0.55       300

    accuracy                           0.49       900
   macro avg       0.49      0.49      0.49       900
weighted avg       0.49      0.49      0.49       900

